# GIOŚ - warstwa BRONZE

Cel: zamiana plików XLSX z warstwy RAW na jedną tabelę w formacie długim (1 wiersz = 1 pomiar) i zapis do Parquet, na którym pracuje DuckDB.

Wynik: pliki `pomiary_2023_2025.parquet`, `stacje.parquet` i `stanowiska.parquet` w folderze `data/bronze/`.

## 1. Ścieżki i otwarcie archiwum

Ustalamy katalog projektu (działa z `notebooks/` i z katalogu głównego) oraz otwieramy ZIP z warstwy RAW.

In [1]:
from pathlib import Path
import zipfile
import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.pipeline_config import YEARS, PERIOD, START, END, HOURS
ZIP_PATH = ROOT / 'data' / 'raw' / str(YEARS[0]) / f'pomiary_{YEARS[0]}.zip'
BRONZE_DIR = ROOT / 'data' / 'bronze'

archive = zipfile.ZipFile(ZIP_PATH)

for name in archive.namelist():
    print(name)

2023_PrekursoryZielonka_1g.xlsx
2023_PM25_24g.xlsx
2023_BbF(PM10)_24g.xlsx
2023_SO2_1g.xlsx
2023_SO2_24g.xlsx
2023_PM25_1g.xlsx
2023_PM10_24g.xlsx
2023_PM10_1g.xlsx
2023_O3_1g.xlsx
2023_Pb(PM10)_24g.xlsx
2023_NO_1g.xlsx
2023_NOx_1g.xlsx
2023_NO2_1g.xlsx
2023_NO2_24g.xlsx
2023_Ni(PM10)_24g.xlsx
2023_Jony_PM25_24g.xlsx
2023_IP(PM10)_24g.xlsx
2023_Hg(TGM)_1g.xlsx
2023_formaldehyd_1g.xlsx
2023_Depozycja_1m.xlsx
2023_DBahA(PM10)_24g.xlsx
2023_CO_1g.xlsx
2023_Cd(PM10)_24g.xlsx
2023_C6H6_1g.xlsx
2023_BjF(PM10)_24g.xlsx
2023_BkF(PM10)_24g.xlsx
2023_BaA(PM10)_24g.xlsx
2023_BaP(PM10)_24g.xlsx
2023_As(PM10)_24g.xlsx


## 2. Wczytanie jednego pliku i podział na nagłówek i dane

Pliki GIOŚ mają 6 wierszy nagłówka (Nr, Kod stacji, Wskaźnik, Czas uśredniania, Jednostka, Kod stanowiska), a dalej jeden wiersz na każdą godzinę lub dobę.

Wczytujemy wszystko jako tekst (`infer_schema_length=0`).

In [2]:
name = f'{YEARS[0]}_PM10_1g.xlsx'

raw = pl.read_excel(
    archive.read(name),
    has_header=False,
    infer_schema_length=0,
)

header = raw.head(6)
data = raw.slice(6)

print('cały plik:', raw.shape)
print(header)
print(data.head(3))

cały plik: (8766, 177)
shape: (6, 177)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ column_0  ┆ column_1  ┆ column_2  ┆ column_3  ┆ … ┆ column_17 ┆ column_17 ┆ column_17 ┆ column_1 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ 3         ┆ 4         ┆ 5         ┆ 76       │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Nr        ┆ 1         ┆ 2         ┆ 3         ┆ … ┆ 173       ┆ 174       ┆ 175       ┆ 176      │
│ Kod       ┆ DsCzerStr ┆ DsDziePil ┆ DsJelGorO ┆ … ┆ ZpSwinMat ┆ ZpSzczAnd ┆ ZpSzczecP ┆ ZpSzczPi │
│ stacji    ┆ aza       ┆ sud       ┆ gin       ┆   ┆ ejkMOB    ┆ rze       ┆ rze       ┆ lsud     │
│ Wskaźnik  ┆ PM10      ┆ PM10      ┆ PM10      ┆ … 

## 3. Opis kolumn stanowisk

Każda kolumna `column_1…column_N` to jedno stanowisko pomiarowe. Obracamy nagłówek (`transpose`), żeby każde stanowisko było jednym wierszem z kodem stacji, wskaźnikiem i jednostką.

In [3]:
columns_info = (
    header
    .drop('column_0')
    .transpose(include_header=True, header_name='kolumna')
)
columns_info.columns = [
    'kolumna', 'nr', 'kod_stacji', 'wskaznik',
    'czas_usredniania', 'jednostka', 'kod_stanowiska',
]

print(columns_info.head())

shape: (5, 7)
┌──────────┬─────┬──────────────┬──────────┬──────────────────┬───────────┬──────────────────────┐
│ kolumna  ┆ nr  ┆ kod_stacji   ┆ wskaznik ┆ czas_usredniania ┆ jednostka ┆ kod_stanowiska       │
│ ---      ┆ --- ┆ ---          ┆ ---      ┆ ---              ┆ ---       ┆ ---                  │
│ str      ┆ str ┆ str          ┆ str      ┆ str              ┆ str       ┆ str                  │
╞══════════╪═════╪══════════════╪══════════╪══════════════════╪═══════════╪══════════════════════╡
│ column_1 ┆ 1   ┆ DsCzerStraza ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsCzerStraza-PM10-1g │
│ column_2 ┆ 2   ┆ DsDziePilsud ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsDziePilsud-PM10-1g │
│ column_3 ┆ 3   ┆ DsJelGorOgin ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsJelGorOgin-PM10-1g │
│ column_4 ┆ 4   ┆ DsKlodzSzkol ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsKlodzSzkol-PM10-1g │
│ column_5 ┆ 5   ┆ DsLegAlRzecz ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsLegAlRzecz-PM10-1

## 4. Zamiana formatu szerokiego na długi

Każdy wiersz wynikowy to pomiar jednego stanowiska w jednej godzinie lub dobie. Liczba godzin zależy od roku.

In [4]:
long = (
    data
    .rename({'column_0': 'data_czas'})
    .unpivot(index='data_czas', variable_name='kolumna', value_name='wartosc')
    .join(columns_info, on='kolumna', how='left')
)

print(long.shape)
print(long.head())

(1541760, 9)
shape: (5, 9)
┌──────────────┬──────────┬─────────┬─────┬───┬──────────┬──────────────┬───────────┬──────────────┐
│ data_czas    ┆ kolumna  ┆ wartosc ┆ nr  ┆ … ┆ wskaznik ┆ czas_usredni ┆ jednostka ┆ kod_stanowis │
│ ---          ┆ ---      ┆ ---     ┆ --- ┆   ┆ ---      ┆ ania         ┆ ---       ┆ ka           │
│ str          ┆ str      ┆ str     ┆ str ┆   ┆ str      ┆ ---          ┆ str       ┆ ---          │
│              ┆          ┆         ┆     ┆   ┆          ┆ str          ┆           ┆ str          │
╞══════════════╪══════════╪═════════╪═════╪═══╪══════════╪══════════════╪═══════════╪══════════════╡
│ 2023-01-01   ┆ column_1 ┆ null    ┆ 1   ┆ … ┆ PM10     ┆ 1g           ┆ ug/m3     ┆ DsCzerStraza │
│ 01:00:00     ┆          ┆         ┆     ┆   ┆          ┆              ┆           ┆ -PM10-1g     │
│ 2023-01-01   ┆ column_1 ┆ null    ┆ 1   ┆ … ┆ PM10     ┆ 1g           ┆ ug/m3     ┆ DsCzerStraza │
│ 02:00:00     ┆          ┆         ┆     ┆   ┆          ┆      

## 5. Typy kolumn i pochodzenie danych

Wybieramy docelowe kolumny i nadajemy typy: `data_czas` na datę (format `%Y-%m-%d %H:%M:%S`), `wartosc` na liczbę (tekst zamiast liczby zatrzyma kod). Dodajemy `plik_zrodlowy` z nazwą pliku XLSX.

Kolumny pomocnicze (`kolumna`, `nr`) usuwamy, bo służyły tylko do połączenia.

In [5]:
bronze = long.select(
    'kod_stanowiska',
    'kod_stacji',
    'wskaznik',
    'czas_usredniania',
    'jednostka',
    pl.col('data_czas').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('wartosc').cast(pl.Float64),
    pl.lit(name).alias('plik_zrodlowy'),
)

print(bronze.schema)
print(bronze.head(3))

Schema({'kod_stanowiska': String, 'kod_stacji': String, 'wskaznik': String, 'czas_usredniania': String, 'jednostka': String, 'data_czas': Datetime(time_unit='us', time_zone=None), 'wartosc': Float64, 'plik_zrodlowy': String})
shape: (3, 8)
┌─────────────┬────────────┬──────────┬────────────┬───────────┬────────────┬─────────┬────────────┐
│ kod_stanowi ┆ kod_stacji ┆ wskaznik ┆ czas_usred ┆ jednostka ┆ data_czas  ┆ wartosc ┆ plik_zrodl │
│ ska         ┆ ---        ┆ ---      ┆ niania     ┆ ---       ┆ ---        ┆ ---     ┆ owy        │
│ ---         ┆ str        ┆ str      ┆ ---        ┆ str       ┆ datetime[μ ┆ f64     ┆ ---        │
│ str         ┆            ┆          ┆ str        ┆           ┆ s]         ┆         ┆ str        │
╞═════════════╪════════════╪══════════╪════════════╪═══════════╪════════════╪═════════╪════════════╡
│ DsCzerStraz ┆ DsCzerStra ┆ PM10     ┆ 1g         ┆ ug/m3     ┆ 2023-01-01 ┆ null    ┆ 2023_PM10_ │
│ a-PM10-1g   ┆ za         ┆          ┆            ┆ 

## 6. Konwersja wszystkich roczników

Powyższy przykład pokazuje jeden arkusz pierwszego roku. Właściwy import wykonuje scripts/bronze_gios.py dla wszystkich lat z konfiguracji. Obsługuje też dodatkowy wiersz nagłówka NO2 2023 oraz ułamki sekund w 2025. Depozycję pomijamy ze względu na inny format i rodzaj pomiaru.

In [6]:
archive.close()
from scripts import bronze_gios as bronze_module
POMIARY_PATH = bronze_module.main()
bronze = pl.read_parquet(POMIARY_PATH)
print('RAZEM:', bronze.shape)
print(bronze.null_count())

2023_PrekursoryZielonka_1g.xlsx 254040


2023_PM25_24g.xlsx 19710


2023_BbF(PM10)_24g.xlsx 6935


2023_SO2_1g.xlsx 744600


2023_SO2_24g.xlsx 1095


2023_PM25_1g.xlsx 805920


... (78 outputs left out)

2025_BbF(PM10)_24g.xlsx 6935


2025_BjF(PM10)_24g.xlsx 6935


Zapisano: data/bronze/pomiary_2023_2025.parquet


RAZEM: (27059646, 8)
shape: (1, 8)
┌─────────────┬────────────┬──────────┬─────────────┬───────────┬───────────┬─────────┬────────────┐
│ kod_stanowi ┆ kod_stacji ┆ wskaznik ┆ czas_usredn ┆ jednostka ┆ data_czas ┆ wartosc ┆ plik_zrodl │
│ ska         ┆ ---        ┆ ---      ┆ iania       ┆ ---       ┆ ---       ┆ ---     ┆ owy        │
│ ---         ┆ u32        ┆ u32      ┆ ---         ┆ u32       ┆ u32       ┆ u32     ┆ ---        │
│ u32         ┆            ┆          ┆ u32         ┆           ┆           ┆         ┆ u32        │
╞═════════════╪════════════╪══════════╪═════════════╪═══════════╪═══════════╪═════════╪════════════╡
│ 0           ┆ 0          ┆ 0        ┆ 0           ┆ 0         ┆ 0         ┆ 1219834 ┆ 0          │
└─────────────┴────────────┴──────────┴─────────────┴───────────┴───────────┴─────────┴────────────┘


## 7. Plik wszystkich pomiarów

Roczne pliki są łączone w jeden Parquet dla całego zakresu.

In [7]:
print('Zapisano:', POMIARY_PATH.relative_to(ROOT).as_posix())
print(f'Rozmiar: {POMIARY_PATH.stat().st_size / 1_000_000:.1f} MB')

Zapisano: data/bronze/pomiary_2023_2025.parquet
Rozmiar: 73.8 MB


## 8. Metadane stacji i stanowisk

Arkusze STACJE i STANOWISKA z pliku metadanych zapisujemy jako `stacje.parquet` i `stanowiska.parquet`. Zmieniamy nazwy kolumn i nadajemy typy (daty, współrzędne). Duplikaty i wartości `BRAK DANYCH` zostają bez zmian.

In [8]:
STACJE_XLSX = ROOT / 'data' / 'raw' / 'stacje' / 'metadane_stacji.xlsx'

stacje = bronze_module.stations_to_bronze(STACJE_XLSX)
stanowiska = bronze_module.positions_to_bronze(STACJE_XLSX)

stacje.write_parquet(BRONZE_DIR / 'stacje.parquet')
stanowiska.write_parquet(BRONZE_DIR / 'stanowiska.parquet')

print('stacje:', stacje.shape)
print('stanowiska:', stanowiska.shape)
print(stacje.head(3))

stacje: (1148, 14)
stanowiska: (5849, 10)
shape: (3, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ kod_miedz ┆ nazwa_sta ┆ stary_kod ┆ … ┆ miejscowo ┆ adres     ┆ szerokosc ┆ dlugosc  │
│ i         ┆ ynarodowy ┆ cji       ┆ _stacji   ┆   ┆ sc        ┆ ---       ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ str       ┆ f64       ┆ f64      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ DsBialka  ┆ null      ┆ Białka    ┆ null      ┆ … ┆ Białka    ┆ null      ┆ 51.197783 ┆ 16.11739 │
│ DsBielGro ┆ null      ┆ Bielawa - ┆ null      ┆ … ┆ Bielawa   ┆ ul. Grota ┆ 50.68251  ┆ 16.61734 │
│ t         ┆           ┆ ul. Grota ┆           ┆   ┆           ┆ Roweckieg ┆           ┆ 8        │
│           ┆           ┆ Roweckie

## 9. Kontrola spójności

Sprawdzamy, czy stacje z pomiarów są w metadanych, czy kody się nie powtarzają i które daty nie dały się odczytać.

In [9]:
kody_pomiary = set(bronze['kod_stacji'].unique())
kody_metadane = set(stacje['kod_stacji'])

print('Stacje z pomiarami w całym okresie:', len(kody_pomiary))
print('Bez wpisu w metadanych:', sorted(kody_pomiary - kody_metadane))
print()
print(
    'Zdublowane kody stacji:',
    stacje.filter(pl.col('kod_stacji').is_duplicated())['kod_stacji'].unique().to_list(),
)
print(
    'Zdublowane kody stanowisk:',
    stanowiska.filter(pl.col('kod_stanowiska').is_duplicated())['kod_stanowiska'].unique().to_list(),
)

print()
for sheet, table in [('STACJE', stacje), ('STANOWISKA', stanowiska)]:
    raw_sheet = pl.read_excel(STACJE_XLSX, sheet_name=sheet, infer_schema_length=0)
    for source_col, col in [
        ('Data uruchomienia', 'data_uruchomienia'),
        ('Data zamknięcia', 'data_zamkniecia'),
    ]:
        filled = raw_sheet[source_col].str.strip_chars().str.len_chars().fill_null(0) > 0
        lost = raw_sheet[source_col].filter(filled & table[col].is_null())
        print(f'{sheet} / {source_col}: nieczytelne daty -> NULL: {lost.to_list()}')

Stacje z pomiarami w całym okresie: 325
Bez wpisu w metadanych: ['DsDusznSportMOB', 'MpZlockieMusMOB', 'OpBrzegPoprzMOB', 'SlGoczaUzdro', 'SlRaciborzWP', 'ZpSzczLaczna']

Zdublowane kody stacji: ['OpKrap3Maja']
Zdublowane kody stanowisk: ['PdSuwPulaski-BaA(PM10)-24g', 'PdSuwPulaski-DBahA(PM10)-24g', 'PdSuwPulaski-IP(PM10)-24g', 'PdSuwPulaski-BbF(PM10)-24g', 'PdSuwPulaski-BjF(PM10)-24g', 'PdSuwPulaski-BkF(PM10)-24g']

STACJE / Data uruchomienia: nieczytelne daty -> NULL: []
STACJE / Data zamknięcia: nieczytelne daty -> NULL: []


STANOWISKA / Data uruchomienia: nieczytelne daty -> NULL: []
STANOWISKA / Data zamknięcia: nieczytelne daty -> NULL: ['31.12.20025', '31.12.20025', '31.12.20025', '31.12.20025']
